# Task 15: 64 维全特征 — ΔΔG + all_3 + TMD

**目的**: 将全部 ΔΔG 特征 (ddg_esm2 + ddg_struct + ddg_rasp + ddg_foldx) 与 TMD 靶向特征 (in_TMD + dist_to_nearest_TMD + delta_membrane_insertion) 联合，看是否能突破 AlphaMissense 基线。

| 配置 | 组成 | 维度 |
|---|---|---|
| **PCA(50) + ΔΔG + all_3** | 50PC + 7struct + ddg_esm2 + ddg_struct + ddg_rasp + ddg_foldx | 61 |
| **PCA(50) + ΔΔG + all_3 + TMD** | 61 + in_TMD + dist_to_nearest_TMD + delta_membrane_insertion | 64 |

**对照**: PCA(50) 基线。

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"

# ===== 加载特征矩阵 =====
df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")
df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")

ID_COLS = ["KEY", "Gene", "reloc_v3"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]
esm_cols = [c for c in feat_cols if c.startswith("esm_")]
STRUCT_NAMES = ["plddt","sasa","rsa","ss_helix","ss_strand","ss_coil",
                "delta_hydrophobicity"]

X_full_arr = df_feat[feat_cols].values.astype(np.float32)
esm_idx = [feat_cols.index(c) for c in esm_cols]
struct_idx = [feat_cols.index(c) for c in STRUCT_NAMES if c in feat_cols]
X_esm = X_full_arr[:, esm_idx]
X_struct = X_full_arr[:, struct_idx]

y_bin = (df_feat["reloc_v3"].values > 0).astype(int)
groups = df_feat["Gene"].values
full_keys = (df_main["Gene"] + "||" + df_main["Mutation_used"]).tolist()

print(f"n={len(y_bin)}, pos={int(y_bin.sum())}, base_rate={y_bin.sum()/len(y_bin):.4f}")

# ===== 加载 4 种 ddg =====
DDG_NAMES = ["ddg_esm2", "ddg_struct", "ddg_rasp", "ddg_foldx"]
ddg_files = {
    "ddg_esm2":   ("ddg_esm2.csv",   "ddg_esm2"),
    "ddg_struct": ("ddg_struct.csv", "ddg_struct"),
    "ddg_rasp":   ("ddg_rasp.csv",   "ddg_rasp"),
    "ddg_foldx":  ("ddg_foldx.csv",  "ddg_foldx"),
}

ddg_sources = {}
for name, (fname, dcol) in ddg_files.items():
    path = BASE_PATH + fname
    df_d = pd.read_csv(path)
    ddg_map = dict(zip(df_d["KEY"], df_d[dcol]))
    arr = np.array([ddg_map.get(k, np.nan) for k in full_keys], dtype=np.float32)
    n_valid = np.isfinite(arr).sum()
    ddg_sources[name] = arr.reshape(-1, 1)
    print(f"  {name:12s}: {n_valid}/{len(arr)} 有效")

# ===== 加载 TMD 特征 =====
TMD_NAMES = ["in_TMD", "dist_to_nearest_TMD", "delta_membrane_insertion"]
try:
    df_tmd = pd.read_csv(BASE_PATH + "tmd_features.csv")
    tmd_sources = {}
    for name in TMD_NAMES:
        tmd_map = dict(zip(df_tmd["KEY"], df_tmd[name]))
        arr = np.array([tmd_map.get(k, 0.0) for k in full_keys], dtype=np.float32)
        tmd_sources[name] = arr.reshape(-1, 1)
        print(f"  {name:25s}: {(arr != 0).sum()}/{len(arr)} 非零")
    has_tmd = True
    print(f"\n已加载 {len(ddg_sources)} ddg + {len(tmd_sources)} TMD")
except FileNotFoundError:
    has_tmd = False
    print("\n⚠ tmd_features.csv 未找到，请先运行 task14")
    print("  将仅对比 PCA vs PCA+ΔΔG+all_3")

n=2179, pos=235, base_rate=0.1078
  ddg_esm2    : 2179/2179 有效
  ddg_struct  : 2168/2179 有效
  ddg_rasp    : 2168/2179 有效
  ddg_foldx   : 2166/2179 有效
  in_TMD                   : 151/2179 非零
  dist_to_nearest_TMD      : 664/2179 非零
  delta_membrane_insertion : 151/2179 非零

已加载 4 ddg + 3 TMD


## 15a. CV: PCA vs +ΔΔG+all_3 vs +ΔΔG+all_3+TMD

In [2]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
N_COMP = 50

if has_tmd:
    CONFIGS = ["PCA", "PCA+ddg_all4", "PCA+ddg_all4+TMD"]
else:
    CONFIGS = ["PCA", "PCA+ddg_all4"]
oof = {k: np.zeros(len(y_bin), dtype=np.float32) for k in CONFIGS}

for fold, (tr_idx, te_idx) in enumerate(cv.split(X_full_arr, y_bin, groups)):
    y_tr = y_bin[tr_idx]; y_te = y_bin[te_idx]

    # --- ESM2 → PCA ---
    imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
    Xe_tr = scl_e.fit_transform(imp_e.fit_transform(X_esm[tr_idx])).astype(np.float32)
    Xe_te = scl_e.transform(imp_e.transform(X_esm[te_idx])).astype(np.float32)
    pca = PCA(n_components=N_COMP, random_state=42)
    Xe_tr_pca = pca.fit_transform(Xe_tr).astype(np.float32)
    Xe_te_pca = pca.transform(Xe_te).astype(np.float32)

    # --- Struct ---
    imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
    Xs_tr = scl_s.fit_transform(imp_s.fit_transform(X_struct[tr_idx])).astype(np.float32)
    Xs_te = scl_s.transform(imp_s.transform(X_struct[te_idx])).astype(np.float32)

    X_tr_base = np.hstack([Xe_tr_pca, Xs_tr]).astype(np.float32)
    X_te_base = np.hstack([Xe_te_pca, Xs_te]).astype(np.float32)

    sw = compute_sample_weight("balanced", y_tr)

    def fit_pred(X_tr, X_te):
        m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.5,
                          objective="binary:logistic", eval_metric="aucpr",
                          random_state=42, n_jobs=-1, tree_method="hist")
        m.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
        return m.predict_proba(X_te)[:, 1]

    # ---- PCA 基线 ----
    oof["PCA"][te_idx] = fit_pred(X_tr_base, X_te_base)

    # ---- + 全部 4 种 ddg ----
    ddg_parts_tr = []; ddg_parts_te = []
    for name in DDG_NAMES:
        imp_d = SimpleImputer(strategy="median")
        ddg_parts_tr.append(imp_d.fit_transform(ddg_sources[name][tr_idx]).astype(np.float32))
        ddg_parts_te.append(imp_d.transform(ddg_sources[name][te_idx]).astype(np.float32))
    oof["PCA+ddg_all4"][te_idx] = fit_pred(
        np.hstack([X_tr_base] + ddg_parts_tr).astype(np.float32),
        np.hstack([X_te_base] + ddg_parts_te).astype(np.float32))

    # ---- + 全部 ddg + TMD ----
    if has_tmd:
        tmd_parts_tr = []; tmd_parts_te = []
        for name in TMD_NAMES:
            imp_t = SimpleImputer(strategy="median")
            tmd_parts_tr.append(imp_t.fit_transform(tmd_sources[name][tr_idx]).astype(np.float32))
            tmd_parts_te.append(imp_t.transform(tmd_sources[name][te_idx]).astype(np.float32))
        oof["PCA+ddg_all4+TMD"][te_idx] = fit_pred(
            np.hstack([X_tr_base] + ddg_parts_tr + tmd_parts_tr).astype(np.float32),
            np.hstack([X_te_base] + ddg_parts_te + tmd_parts_te).astype(np.float32))

    vals = {k: roc_auc_score(y_te, oof[k][te_idx]) for k in CONFIGS}
    print(f"  Fold {fold}: " + "  ".join([f"{k}={v:.4f}" for k, v in vals.items()]))

  Fold 0: PCA=0.6606  PCA+ddg_all4=0.7068  PCA+ddg_all4+TMD=0.7429
  Fold 1: PCA=0.6129  PCA+ddg_all4=0.6200  PCA+ddg_all4+TMD=0.6920
  Fold 2: PCA=0.5456  PCA+ddg_all4=0.5845  PCA+ddg_all4+TMD=0.5628
  Fold 3: PCA=0.6049  PCA+ddg_all4=0.5978  PCA+ddg_all4+TMD=0.6484
  Fold 4: PCA=0.6202  PCA+ddg_all4=0.5969  PCA+ddg_all4+TMD=0.6215


## 15b. 汇总

In [3]:
br = float(y_bin.sum() / len(y_bin))
results = {k: {"auroc": roc_auc_score(y_bin, oof[k]),
                "auprc": average_precision_score(y_bin, oof[k])}
           for k in CONFIGS}
auroc_pca = results["PCA"]["auroc"]

labels = {"PCA": "PCA(50) 基线",
          "PCA+ddg_all4": "+ ΔΔG + all_3 (4×ddG)",
          "PCA+ddg_all4+TMD": "+ 4×ddG + 3×TMD (64维)"}
dims = {"PCA": 57, "PCA+ddg_all4": 61, "PCA+ddg_all4+TMD": 64}

print(f"\n{'='*85}")
print(f"  Task 15: 64 维全特征 (n={len(y_bin)}, pos={int(y_bin.sum())}, base_rate={br:.4f})")
print(f"  {'配置':<45s} {'AUROC':>8s} {'AUPRC':>8s} {'ΔAUROC':>12s} {'维度':>6s}")
print(f"  {'-'*80}")

for k in CONFIGS:
    r = results[k]
    d = r["auroc"] - auroc_pca
    m = " +" if d > 0.005 else ("" if d > -0.005 else " -")
    print(f"  {labels[k]:<45s} {r['auroc']:>8.4f} {r['auprc']:>8.4f} {d:>+12.4f}{m} {dims[k]:>5d}")

print(f"{'='*85}")

# 增益
d_ddg = results["PCA+ddg_all4"]["auroc"] - auroc_pca
if has_tmd:
    d_tmd = results["PCA+ddg_all4+TMD"]["auroc"] - auroc_pca
    d_tmd_alone = results["PCA+ddg_all4+TMD"]["auroc"] - results["PCA+ddg_all4"]["auroc"]
    print(f"\n增益分解:")
    print(f"  4×ddG 增益:     {d_ddg:+.4f}")
    print(f"  +TMD 额外增益:  {d_tmd_alone:+.4f}")
    print(f"  总增益:         {d_tmd:+.4f}")

best = max(r["auroc"] for r in results.values())
best_name = [k for k, r in results.items() if r["auroc"] == best][0]
print(f"\n最佳: {labels[best_name]} AUROC={best:.4f}")
print(f"AlphaMissense: 0.6374, gap={0.6374 - best:.4f}")


  Task 15: 64 维全特征 (n=2179, pos=235, base_rate=0.1078)
  配置                                               AUROC    AUPRC       ΔAUROC     维度
  --------------------------------------------------------------------------------
  PCA(50) 基线                                      0.6087   0.1597      +0.0000    57
  + ΔΔG + all_3 (4×ddG)                           0.6222   0.1778      +0.0135 +    61
  + 4×ddG + 3×TMD (64维)                           0.6546   0.2026      +0.0459 +    64

增益分解:
  4×ddG 增益:     +0.0135
  +TMD 额外增益:  +0.0324
  总增益:         +0.0459

最佳: + 4×ddG + 3×TMD (64维) AUROC=0.6546
AlphaMissense: 0.6374, gap=-0.0172


## 15c. 特征重要性 (64 维全特征, fit on all data)

In [4]:
if not has_tmd:
    print("跳过: 需要 tmd_features.csv")
else:
    print("\n--- 特征重要性 PCA(50)+4ddG+3TMD (64维) ---")

    # PCA on all data
    imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
    Xe_all = scl_e.fit_transform(imp_e.fit_transform(X_esm)).astype(np.float32)
    pca_all = PCA(n_components=N_COMP, random_state=42)
    Xe_pca_all = pca_all.fit_transform(Xe_all).astype(np.float32)

    imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
    Xs_all = scl_s.fit_transform(imp_s.fit_transform(X_struct)).astype(np.float32)

    parts = [Xe_pca_all, Xs_all]
    all_names = [f"PC{i+1}" for i in range(N_COMP)] + STRUCT_NAMES

    for name in DDG_NAMES:
        imp_d = SimpleImputer(strategy="median")
        parts.append(imp_d.fit_transform(ddg_sources[name]).astype(np.float32))
        all_names.append(name)
    for name in TMD_NAMES:
        imp_t = SimpleImputer(strategy="median")
        parts.append(imp_t.fit_transform(tmd_sources[name]).astype(np.float32))
        all_names.append(name)

    X_all = np.hstack(parts).astype(np.float32)

    sw_ratio = (y_bin==0).sum() / max((y_bin==1).sum(), 1)
    xgb_fi = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.5,
                           scale_pos_weight=sw_ratio,
                           objective="binary:logistic", eval_metric="aucpr",
                           random_state=42, n_jobs=-1, tree_method="hist")
    xgb_fi.fit(X_all, y_bin, verbose=False)

    imps = xgb_fi.feature_importances_
    idx_sorted = np.argsort(imps)[::-1]

    print("Top-20 特征:")
    for rank, i in enumerate(idx_sorted[:20]):
        val = imps[i]; bar = "\u2588" * int(val * 2000)
        print(f"  {rank+1:2d}. {all_names[i]:30s}  {val:.5f}  {bar}")

    # ddG 排名
    print(f"\n四种 ddG 排名 (共 {len(all_names)} 个特征):")
    for name in DDG_NAMES:
        i = all_names.index(name)
        rank = int(np.where(idx_sorted == i)[0][0]) + 1
        marker = " \u2605" if rank <= 10 else (" \u2605 top-20!" if rank <= 20 else "")
        print(f"  {name:15s} 排名={rank:3d}/{len(all_names)}  重要性={imps[i]:.5f}{marker}")

    # TMD 排名
    print(f"\nTMD 特征排名:")
    for name in TMD_NAMES:
        i = all_names.index(name)
        rank = int(np.where(idx_sorted == i)[0][0]) + 1
        marker = " \u2605 top-10!" if rank <= 10 else (" \u2605 top-20!" if rank <= 20 else "")
        print(f"  {name:25s} 排名={rank:3d}/{len(all_names)}  重要性={imps[i]:.5f}{marker}")

    # 结构特征排名
    print(f"\n结构特征排名:")
    for name in STRUCT_NAMES:
        i = all_names.index(name)
        rank = int(np.where(idx_sorted == i)[0][0]) + 1
        marker = " \u2605 top-10!" if rank <= 10 else (" \u2605 top-20!" if rank <= 20 else "")
        print(f"  {name:25s} 排名={rank:3d}/{len(all_names)}  重要性={imps[i]:.5f}{marker}")


--- 特征重要性 PCA(50)+4ddG+3TMD (64维) ---
Top-20 特征:
   1. ddg_esm2                        0.03583  ███████████████████████████████████████████████████████████████████████
   2. dist_to_nearest_TMD             0.03300  █████████████████████████████████████████████████████████████████
   3. PC1                             0.02260  █████████████████████████████████████████████
   4. ss_coil                         0.02174  ███████████████████████████████████████████
   5. PC36                            0.02149  ██████████████████████████████████████████
   6. delta_membrane_insertion        0.02075  █████████████████████████████████████████
   7. PC39                            0.02037  ████████████████████████████████████████
   8. ddg_rasp                        0.02015  ████████████████████████████████████████
   9. PC42                            0.01946  ██████████████████████████████████████
  10. in_TMD                          0.01922  ██████████████████████████████████████
  11. P